In [1]:
include("../RayTracing.jl")

Main.RayTracing

In [ ]:
parsed_args = RayTracing.parse_commandline()
    
# set up logging
logger = RayTracing.setup_logging(parsed_args["debug"])
RayTracing.global_logger(logger)

# set random seed
RayTracing.Random.seed!(parsed_args["seed"])

BASE_FNAME = "hehe.exr"

parsed_args["image-dim"] = [150, 150]
parsed_args["samples-per-pixel"] = 16

16

In [3]:
function unit_circle_points_rgb(
    N::Int, 
    colors::Vector{Tuple{Float64, Float64, Float64}},
    x_shift::Real=0.0, 
    y_shift::Real=0.0, 
    x_scale::Real=1.0, 
    y_scale::Real=1.0
)
    # Check for valid input
    if N <= 0
        throw(ArgumentError("Number of points must be positive"))
    end
    
    if isempty(colors)
        throw(ArgumentError("Colors list cannot be empty"))
    end
    
    # Calculate angular spacing in radians (2π divided by N)
    θ_step = 2π / N
    
    # Generate the points with colors
    points = Vector{Tuple{Float64, Float64, Tuple{Float64, Float64, Float64}}}(undef, N)
    
    for i in 1:N
        # Calculate the angle for this point
        θ = (i - 1) * θ_step
        
        # Convert to Cartesian coordinates, apply scaling and shifting
        x = cos(θ) * x_scale + x_shift
        y = sin(θ) * y_scale + y_shift
        
        # Calculate color for this point
        # Map the current position (0 to N-1) to a position in the color list (0 to length(colors)-1)
        color_position = (i - 1) * (length(colors) - 1) / (N - 1)
        
        # Get the indices of the two colors to interpolate between
        color_index_low = floor(Int, color_position) + 1
        color_index_high = min(color_index_low + 1, length(colors))
        
        # Get the two colors
        color_low = colors[color_index_low]
        color_high = colors[color_index_high]
        
        # Calculate the interpolation factor (0 to 1)
        t = color_position - floor(color_position)
        
        # If we're at the exact position of a color in the list, use that color
        if color_index_low == color_index_high
            rgb = color_low
        else
            # Interpolate between the two colors
            r = color_low[1] * (1 - t) + color_high[1] * t
            g = color_low[2] * (1 - t) + color_high[2] * t
            b = color_low[3] * (1 - t) + color_high[3] * t
            
            rgb = (r, g, b)
        end
        
        # Store the point with its color
        points[i] = (x, y, rgb)
    end
    
    return points
end

unit_circle_points_rgb (generic function with 5 methods)

In [8]:
# manually build scene
colors = [
    (1.3, 0.3, 0.3),  # Red-ish
    (0.3, 1.3, 0.3),  # Green-ish
    (0.3, 0.3, 1.3),  # Blue-ish
    (1.3, 1.3, 0.3)   # Yellow-ish
]
circle_output = unit_circle_points_rgb(16, colors, 0.0, 1.9, .5, .5)

for (i, (shear_c, scale_y, (r, g, b))) in enumerate(circle_output)
    parsed_args["file-name"] = replace(BASE_FNAME, ".exr" => "_$(lpad(string(i),2,"0")).exr")
    primitives = RayTracing.Primitive[]
    lights = RayTracing.Light[]

    old_smile = RayTracing.jmfp("/Users/johnmyslinski/Documents/PBRJ/ref/smile3.png")
    new_smile = RayTracing.jmfp("/Users/johnmyslinski/Documents/PBRJ/ref/smile3_post.exr")

    # make my PNG blue!
    RayTracing.party_blob_fuckery!(
        old_smile,
        new_smile,
        (r, g, b)
    )

    # materials
    mat_gray = RayTracing.Matte(
        RayTracing.ConstantTexture(RayTracing.spectrum_from_float(0.5, 0.5, 0.5)),
        RayTracing.ConstantTexture(RayTracing.spectrum_from_float(0.0, 0.0, 0.0)),
        nothing
    )

    emissive_color = RayTracing.spectrum_from_float(r, g, b)
    Kd = RayTracing.MixMultTexture(
        RayTracing.ConstantTexture(emissive_color),
        RayTracing.ImageTexture(RayTracing.UVMapping2D(), new_smile)
    )
    mat_blob = RayTracing.Matte(
        Kd,
        RayTracing.ConstantTexture(RayTracing.spectrum_from_float(0.0, 0.0, 0.0)),
        nothing
    )

    ###############
    ### a thing ###
    ###############       

    radius = 1.0
    sphere_t = RayTracing.Shear(0.0, 0.0, shear_c, 0.0, 0.0, 0.0) * RayTracing.Scale(1.0, scale_y, 1.0) * RayTracing.RotateY(130.0) * RayTracing.RotateZ(-35.0) * RayTracing.RotateX(-80.0)
    sphere = RayTracing.Sphere(
        RayTracing.ShapeCore(sphere_t, RayTracing.Inv(sphere_t), false, false),
        radius
    )
    alight = RayTracing.DiffuseAreaLight(
        emissive_color,
        sphere,
        false,
        nothing,
        new_smile,
        1.0
    )
    push!(primitives, RayTracing.Primitive(sphere, mat_blob, alight))
    push!(lights, alight)

    floor_transform = RayTracing.Translate(RayTracing.Pnt3(0,0,0))
    floor = RayTracing.Rectangle(
        RayTracing.Pnt2(-10, -10),
        RayTracing.Pnt2(10, 10),
        0.0,
        2, 
        RayTracing.ShapeCore(floor_transform, RayTracing.Inv(floor_transform), false, false),
        false,
        nothing
    )
    for tri in floor
        push!(primitives, RayTracing.Primitive(tri, mat_gray, nothing))
    end

    # instantiate accelerator
    print("\nThere are " * RayTracing.num2str(length(primitives)) * " objects in the scene, building BVH\n")
    @time bvh = RayTracing.BVH(primitives)
    print("Done building BVH\n")

    # Instantiate a Filter
    filter = RayTracing.BoxFilter(RayTracing.Pnt2(.5, .5))

    # Instantiate a Film
    film = RayTracing.Film(
        RayTracing.Pnt2(parsed_args["image-dim"][1], parsed_args["image-dim"][2]),
        RayTracing.Bounds2(RayTracing.Pnt2(parsed_args["crop-window"][1], parsed_args["crop-window"][2]), RayTracing.Pnt2(parsed_args["crop-window"][3], parsed_args["crop-window"][4])),
        filter,
        1.0,
        1.0,
        parsed_args["file-name"]
    )

    # Instantiate a Camera
    look_from = RayTracing.Pnt3(4, 4, 4)
    look_at = RayTracing.Pnt3(0, radius, 0)
    up = RayTracing.Vec3(0, 1, 0)
    screen = RayTracing.Bounds2(RayTracing.Pnt2(-1, -1), RayTracing.Pnt2(1, 1))
    C = RayTracing.PerspectiveCamera(RayTracing.LookAt(look_from, look_at, up), screen, 0.0, 1.0, 0.0, 1e6, 37.0, film)

    # Instantiate a Sampler
    S = RayTracing.ZSobolSampler(
        parsed_args["samples-per-pixel"], 
        RayTracing.Pnt2(parsed_args["image-dim"][1], parsed_args["image-dim"][2]), 
        Int8(2)
    )
    print("Using " * RayTracing.num2str(S.samples_per_pixel) * " samples per pixel\n")
    
    # Instantiate Scene
    print("There are " * RayTracing.num2str(length(lights)) * " lights in the scene\n")
    scene = RayTracing.Scene(lights, bvh)
    
    # Instantiate an Integrator
    I = RayTracing.BDPTIntegrator(C, S, parsed_args["max-depth"])

    image = RayTracing.render(
        I, 
        scene, 
        parsed_args,
        (-1, -1)
    )
    RayTracing.OpenEXR.save(I.camera.core.core.film.filename, image)
end




There are 3 objects in the scene, building BVH
  0.072142 seconds (136.66 k allocations: 9.196 MiB, 99.92% compilation time)
Done building BVH
Using 16 samples per pixel
There are 1 lights in the scene
Rendering 100 tiles


Progress:   0%|                                         |  ETA: N/A

Utilizing 1 threads



Progress: 100%|█████████████████████████████████████████| Time: 0:00:25



There are 3 objects in the scene, building BVH
  0.000009 seconds (16 allocations: 1.188 KiB)
Done building BVH
Using 16 samples per pixel
There are 1 lights in the scene
Rendering 100 tiles
Utilizing 1 threads



...

Progress: 100%|█████████████████████████████████████████| Time: 0:00:14



There are 3 objects in the scene, building BVH
  0.000010 seconds (16 allocations: 1.188 KiB)
Done building BVH
Using 16 samples per pixel
There are 1 lights in the scene
Rendering 100 tiles
Utilizing 1 threads



Progress: 100%|█████████████████████████████████████████| Time: 0:00:14



There are 3 objects in the scene, building BVH
  0.000010 seconds (16 allocations: 1.188 KiB)
Done building BVH
Using 16 samples per pixel
There are 1 lights in the scene
Rendering 100 tiles
Utilizing 1 threads



Progress: 100%|█████████████████████████████████████████| Time: 0:00:15


In [6]:
# then run this to convery to gif
# convert -delay 1 -loop 0 -set colorspace RGB -colorspace sRGB *.exr barty-plob.gif